<a href="https://colab.research.google.com/github/Sasidhar015/Data-engineering-training-2026/blob/main/30-10-2026_ASSESMENTS_PySpark/Assesment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Assessment - 2**

Streaming Platform Analytics

In [ ]:
!pip -q install pyspark

from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
import os

uploaded = files.upload()  # Select all six CSV files

spark = (SparkSession.builder
         .appName("StreamingSet4")
         .getOrCreate())

def read_csv(name):
    return spark.read.csv(name, header=True, inferSchema=True)

sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")

print("All six datasets loaded successfully.")

Saving content_catalog_120.csv to content_catalog_120.csv
Saving device_master_8.csv to device_master_8.csv
Saving subscriber_master_2400.csv to subscriber_master_2400.csv
Saving subscription_plans_4.csv to subscription_plans_4.csv
Saving watch_sessions_80000.csv to watch_sessions_80000.csv
Saving watch_sessions_extra_10000.csv to watch_sessions_extra_10000.csv
All six datasets loaded successfully.


In [ ]:
print("Main session row count:", sessions_df.count())
sessions_df.printSchema()
sessions_df.show(10, truncate=False)

Main session row count: 80000
root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+-----

In [ ]:
selected_sessions_df = sessions_df.select(
    "session_id", "subscriber_id", "content_id",
    "watch_minutes", "session_status")

completed_over_60_df = selected_sessions_df.filter(
    (F.col("session_status") == "Completed") &
    (F.col("watch_minutes") > 60))

completed_over_60_df.show(20, truncate=False)

+----------+-------------+----------+-------------+--------------+
|session_id|subscriber_id|content_id|watch_minutes|session_status|
+----------+-------------+----------+-------------+--------------+
|WS00000049|SUB01859     |CT0075    |125          |Completed     |
|WS00000179|SUB00935     |CT0054    |62           |Completed     |
|WS00000184|SUB00072     |CT0119    |91           |Completed     |
|WS00000222|SUB01603     |CT0103    |116          |Completed     |
|WS00000233|SUB01233     |CT0027    |158          |Completed     |
|WS00000244|SUB00985     |CT0048    |140          |Completed     |
|WS00000249|SUB00681     |CT0100    |120          |Completed     |
|WS00000257|SUB02115     |CT0120    |68           |Completed     |
|WS00000292|SUB01068     |CT0048    |141          |Completed     |
|WS00000326|SUB00368     |CT0096    |97           |Completed     |
|WS00000415|SUB01758     |CT0105    |150          |Completed     |
|WS00000419|SUB00367     |CT0036    |99           |Completed  

In [ ]:
engagement_sessions_df = (
    sessions_df
    .withColumn("watch_hours", F.round(F.col("watch_minutes") / 60.0, 3))
    .withColumn(
        "high_engagement",
        F.when(F.col("completion_pct").isNull(), F.lit(None).cast("boolean"))
         .when(F.col("completion_pct") >= 75, F.lit(True))
         .otherwise(F.lit(False))))

engagement_sessions_df.select(
    "session_id", "watch_minutes", "watch_hours",
    "completion_pct", "high_engagement").show(15, truncate=False)

+----------+-------------+-----------+--------------+---------------+
|session_id|watch_minutes|watch_hours|completion_pct|high_engagement|
+----------+-------------+-----------+--------------+---------------+
|WS00000001|6            |0.1        |11.8          |false          |
|WS00000002|31           |0.517      |56.4          |false          |
|WS00000003|29           |0.483      |63.0          |false          |
|WS00000004|69           |1.15       |75.0          |true           |
|WS00000005|36           |0.6        |23.4          |false          |
|WS00000006|20           |0.333      |19.8          |false          |
|WS00000007|78           |1.3        |83.9          |true           |
|WS00000008|22           |0.367      |66.7          |false          |
|WS00000009|21           |0.35       |80.8          |true           |
|WS00000010|21           |0.35       |65.6          |false          |
|WS00000011|22           |0.367      |36.1          |false          |
|WS00000012|81      

In [ ]:
top_data_usage_df = sessions_df.orderBy(F.col("data_used_mb").desc_nulls_last())
top_data_usage_df.show(20, truncate=False)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|session_start      |watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00051169|SUB00155     |CT0069    |DV08     |2026-07-20 00:11:00|167          |97.7          |9.0              |4K            |Completed     |8881.8      |3.0        |
|WS00074513|SUB00684     |CT0056    |DV05     |2026-08-09 21:53:00|157          |96.9          |8.0              |4K            |Completed     |8408.4      |NULL       |
|WS00039201|SUB01664     |CT0027    |DV08     |2026-05-16 06:35:00|161          |97.0          |9.0              |4K            |Completed     |8282.0

In [ ]:
missing_values_df = sessions_df.select([
    F.sum(F.col(c).isNull().cast("long")).alias(c + "_null_count")
    for c in [
        "subscriber_id", "device_id", "completion_pct",
        "buffering_seconds", "user_rating"]])

missing_values_df.show(truncate=False)

+------------------------+--------------------+-------------------------+----------------------------+----------------------+
|subscriber_id_null_count|device_id_null_count|completion_pct_null_count|buffering_seconds_null_count|user_rating_null_count|
+------------------------+--------------------+-------------------------+----------------------------+----------------------+
|951                     |2003                |3513                     |4424                        |77605                 |
+------------------------+--------------------+-------------------------+----------------------------+----------------------+



In [ ]:
sessions_buffering_filled_df = sessions_df.fillna({
    "buffering_seconds": 0})

print("Original DataFrame is unchanged.")
sessions_buffering_filled_df.select(
    "session_id", "buffering_seconds").show(10, truncate=False)

Original DataFrame is unchanged.
+----------+-----------------+
|session_id|buffering_seconds|
+----------+-----------------+
|WS00000001|29.0             |
|WS00000002|31.0             |
|WS00000003|26.0             |
|WS00000004|2.0              |
|WS00000005|31.0             |
|WS00000006|202.0            |
|WS00000007|28.0             |
|WS00000008|11.0             |
|WS00000009|12.0             |
|WS00000010|166.0            |
+----------+-----------------+
only showing top 10 rows


In [ ]:
sessions_with_device_df = sessions_df.filter(F.col("device_id").isNotNull())

sessions_before_count = sessions_df.count()
sessions_after_device_filter_count = sessions_with_device_df.count()

print("Original session count:", sessions_before_count)
print("Rows with non-null device_id:", sessions_after_device_filter_count)
print("Rows excluded due to missing device_id:",
      sessions_before_count - sessions_after_device_filter_count)

Original session count: 80000
Rows with non-null device_id: 77997
Rows excluded due to missing device_id: 2003


In [ ]:
completion_band_df = sessions_df.withColumn(
    "completion_band",
    F.when(F.col("completion_pct").isNull(), "Unknown")
     .when(F.col("completion_pct") < 30, "Low")
     .when(F.col("completion_pct") < 90, "Medium")
     .otherwise("High"))

completion_band_df.select(
    "session_id", "completion_pct", "completion_band").show(20, truncate=False)

+----------+--------------+---------------+
|session_id|completion_pct|completion_band|
+----------+--------------+---------------+
|WS00000001|11.8          |Low            |
|WS00000002|56.4          |Medium         |
|WS00000003|63.0          |Medium         |
|WS00000004|75.0          |Medium         |
|WS00000005|23.4          |Low            |
|WS00000006|19.8          |Low            |
|WS00000007|83.9          |Medium         |
|WS00000008|66.7          |Medium         |
|WS00000009|80.8          |Medium         |
|WS00000010|65.6          |Medium         |
|WS00000011|36.1          |Medium         |
|WS00000012|78.6          |Medium         |
|WS00000013|51.6          |Medium         |
|WS00000014|28.1          |Low            |
|WS00000015|16.4          |Low            |
|WS00000016|28.0          |Low            |
|WS00000017|84.3          |Medium         |
|WS00000018|83.3          |Medium         |
|WS00000019|90.3          |High           |
|WS00000020|27.0          |Low  

In [ ]:
sessions_content_df = sessions_df.join(
    content_df,
    on="content_id",
    how="left")

genre_viewing_summary_df = (
    sessions_content_df
    .groupBy("genre")
    .agg(
        F.count("*").alias("session_count"),
        F.sum("watch_minutes").alias("total_watch_minutes"))
    .orderBy(F.col("total_watch_minutes").desc_nulls_last()))

genre_viewing_summary_df.show(truncate=False)

+-----------+-------------+-------------------+
|genre      |session_count|total_watch_minutes|
+-----------+-------------+-------------------+
|Adventure  |15561        |839700             |
|Drama      |10733        |519872             |
|Animation  |11886        |490335             |
|Comedy     |9105         |452563             |
|Sci-Fi     |9344         |433946             |
|Sports     |9967         |383044             |
|Documentary|7432         |256626             |
|Thriller   |5972         |248430             |
+-----------+-------------+-------------------+



In [ ]:
sessions_device_df = sessions_df.join(
    devices_df,
    on="device_id",
    how="left")

device_performance_df = (
    sessions_device_df
    .groupBy("device_type")
    .agg(
        F.round(F.avg("buffering_seconds"), 2).alias("avg_buffering_seconds"),
        F.round(F.avg("watch_minutes"), 2).alias("avg_watch_minutes"))
    .orderBy(F.col("avg_buffering_seconds").desc_nulls_last()))

device_performance_df.show(truncate=False)

+---------------+---------------------+-----------------+
|device_type    |avg_buffering_seconds|avg_watch_minutes|
+---------------+---------------------+-----------------+
|Laptop Browser |36.93                |45.84            |
|Smart TV       |36.67                |45.34            |
|Streaming Stick|36.32                |45.09            |
|Desktop Browser|36.03                |45.57            |
|iPhone         |35.76                |45.23            |
|Android Phone  |35.68                |45.09            |
|NULL           |35.58                |44.79            |
|Game Console   |35.51                |45.04            |
|Tablet         |35.24                |45.39            |
+---------------+---------------------+-----------------+



In [ ]:
sessions_nonnull_subscriber_df = sessions_df.filter(
    F.col("subscriber_id").isNotNull())

sessions_subscriber_df = sessions_nonnull_subscriber_df.join(
    subscribers_df.select("subscriber_id", "plan_id"),
    on="subscriber_id",
    how="left")

sessions_subscriber_plan_df = sessions_subscriber_df.join(
    plans_df.select("plan_id", "plan_name"),
    on="plan_id",
    how="left")

completion_rate_by_plan_df = (
    sessions_subscriber_plan_df
    .groupBy("plan_name")
    .agg(
        F.count("*").alias("all_sessions"),
        F.sum(F.when(F.col("session_status") == "Completed", 1).otherwise(0))
         .alias("completed_sessions"))
    .withColumn(
        "completion_rate_pct",
        F.round(
            F.col("completed_sessions") * 100.0 / F.col("all_sessions"),2))
    .orderBy(F.col("completion_rate_pct").desc_nulls_last()))

completion_rate_by_plan_df.show(truncate=False)

+---------+------------+------------------+-------------------+
|plan_name|all_sessions|completed_sessions|completion_rate_pct|
+---------+------------+------------------+-------------------+
|Mobile   |17510       |1429              |8.16               |
|Basic    |25086       |2006              |8.0                |
|Premium  |12198       |975               |7.99               |
|Standard |23684       |1868              |7.89               |
|NULL     |571         |42                |7.36               |
+---------+------------+------------------+-------------------+



In [ ]:
active_subscribers_df = subscribers_df.filter(
    F.col("subscriber_status") == "Active")

active_subscriber_plan_df = active_subscribers_df.join(
    plans_df,
    on="plan_id",
    how="inner")
monthly_revenue_by_plan_df = (
    active_subscriber_plan_df
    .groupBy("plan_name", "monthly_fee_inr")
    .agg(F.countDistinct("subscriber_id").alias("active_subscriber_count"))
    .withColumn(
        "estimated_monthly_revenue_inr",
        F.col("active_subscriber_count") * F.col("monthly_fee_inr"))
    .orderBy(F.col("estimated_monthly_revenue_inr").desc()))

monthly_revenue_by_plan_df.show(truncate=False)


+---------+---------------+-----------------------+-----------------------------+
|plan_name|monthly_fee_inr|active_subscriber_count|estimated_monthly_revenue_inr|
+---------+---------------+-----------------------+-----------------------------+
|Standard |399            |604                    |240996                       |
|Premium  |649            |326                    |211574                       |
|Basic    |249            |632                    |157368                       |
|Mobile   |149            |443                    |66007                        |
+---------+---------------+-----------------------+-----------------------------+



In [ ]:
unmatched_subscriber_sessions_df = (
    sessions_df
    .filter(F.col("subscriber_id").isNotNull())
    .join(
        subscribers_df.select("subscriber_id").distinct(),
        on="subscriber_id",
        how="left_anti"))

unmatched_device_sessions_df = (
    sessions_df
    .filter(F.col("device_id").isNotNull())
    .join(
        devices_df.select("device_id").distinct(),
        on="device_id",
        how="left_anti"))

print("Sessions with unmatched non-null subscriber_id:",
      unmatched_subscriber_sessions_df.count())
print("Sessions with unmatched non-null device_id:",
      unmatched_device_sessions_df.count())

Sessions with unmatched non-null subscriber_id: 571
Sessions with unmatched non-null device_id: 481


In [ ]:
sessions_timestamp_df = sessions_df.withColumn(
    "session_start_ts", F.to_timestamp("session_start"))

monthly_2026_summary_df = (
    sessions_timestamp_df
    .filter(F.year("session_start_ts") == 2026)
    .withColumn("session_year", F.year("session_start_ts"))
    .withColumn("session_month", F.month("session_start_ts"))
    .groupBy("session_year", "session_month")
    .agg(
        F.count("*").alias("session_count"),
        F.sum("watch_minutes").alias("total_watch_minutes") )
    .orderBy("session_year", "session_month"))

monthly_2026_summary_df.show(truncate=False)

+------------+-------------+-------------+-------------------+
|session_year|session_month|session_count|total_watch_minutes|
+------------+-------------+-------------+-------------------+
|2026        |1            |9070         |407215             |
|2026        |2            |8218         |372726             |
|2026        |3            |9019         |403971             |
|2026        |4            |8817         |399432             |
|2026        |5            |9006         |407386             |
|2026        |6            |8776         |404203             |
|2026        |7            |9204         |416104             |
|2026        |8            |8996         |407374             |
|2026        |9            |8894         |406105             |
+------------+-------------+-------------+-------------------+



In [ ]:
sessions_with_timestamp_df = sessions_df.withColumn(
    "session_start_ts", F.to_timestamp("session_start"))

subscriber_signup_df = subscribers_df.select(
    "subscriber_id",
    F.to_date("signup_date").alias("signup_date"))

sessions_signup_gap_df = (
    sessions_with_timestamp_df
    .filter(F.col("subscriber_id").isNotNull())
    .join(subscriber_signup_df, on="subscriber_id", how="inner")
    .withColumn(
        "days_from_signup_to_session",
        F.datediff(F.to_date("session_start_ts"), F.col("signup_date"))))

sessions_signup_gap_df.orderBy(
    F.col("session_start_ts").asc()).select(
    "session_id", "subscriber_id", "signup_date",
    "session_start_ts", "days_from_signup_to_session").show(10, truncate=False)

+----------+-------------+-----------+-------------------+---------------------------+
|session_id|subscriber_id|signup_date|session_start_ts   |days_from_signup_to_session|
+----------+-------------+-----------+-------------------+---------------------------+
|WS00035235|SUB01707     |2024-02-13 |2026-01-01 00:00:00|688                        |
|WS00074739|SUB02206     |2025-08-31 |2026-01-01 00:04:00|123                        |
|WS00032058|SUB01248     |2024-03-17 |2026-01-01 00:04:00|655                        |
|WS00048560|SUB02342     |2025-08-25 |2026-01-01 00:14:00|129                        |
|WS00067087|SUB00563     |2024-08-11 |2026-01-01 00:15:00|508                        |
|WS00004245|SUB00711     |2024-10-26 |2026-01-01 00:18:00|432                        |
|WS00041710|SUB00261     |2025-05-15 |2026-01-01 00:21:00|231                        |
|WS00045132|SUB02167     |2025-11-22 |2026-01-01 00:28:00|40                         |
|WS00036438|SUB01237     |2024-02-01 |2026-

In [ ]:
sessions_day_type_df = (
    sessions_timestamp_df
    .withColumn("day_of_week", F.dayofweek("session_start_ts"))
    .withColumn(
        "day_type",
        F.when(F.col("day_of_week").isin(1, 7), "Weekend")
         .otherwise("Weekday")))

weekday_weekend_summary_df = (
    sessions_day_type_df
    .groupBy("day_type")
    .agg(
        F.count("*").alias("session_count"),
        F.round(F.avg("watch_minutes"), 2).alias("avg_watch_minutes"))
    .orderBy("day_type"))

weekday_weekend_summary_df.show(truncate=False)

+--------+-------------+-----------------+
|day_type|session_count|avg_watch_minutes|
+--------+-------------+-----------------+
|Weekday |57228        |45.26            |
|Weekend |22772        |45.41            |
+--------+-------------+-----------------+



In [ ]:
genre_title_watch_df = (
    sessions_content_df
    .groupBy("genre", "content_id", "content_title")
    .agg(F.sum("watch_minutes").alias("total_watch_minutes")))

genre_title_window = Window.partitionBy("genre").orderBy(
    F.col("total_watch_minutes").desc())

top_three_titles_by_genre_df = (
    genre_title_watch_df
    .withColumn("title_rank", F.dense_rank().over(genre_title_window))
    .filter(F.col("title_rank") <= 3)
    .orderBy("genre", "title_rank", "content_title"))

top_three_titles_by_genre_df.show(100, truncate=False)

+-----------+----------+----------------+-------------------+----------+
|genre      |content_id|content_title   |total_watch_minutes|title_rank|
+-----------+----------+----------------+-------------------+----------+
|Adventure  |CT0105    |Infinite Horizon|66395              |1         |
|Adventure  |CT0048    |Lost Journey    |61823              |2         |
|Adventure  |CT0042    |Solar Memory    |60540              |3         |
|Animation  |CT0106    |Last River      |51398              |1         |
|Animation  |CT0114    |Emerald Circuit |51310              |2         |
|Animation  |CT0100    |Golden Puzzle   |50515              |3         |
|Comedy     |CT0027    |Blue Horizon    |61428              |1         |
|Comedy     |CT0019    |Bright Valley   |60921              |2         |
|Comedy     |CT0059    |Silver Valley   |60422              |3         |
|Documentary|CT0077    |Fading Signal   |47085              |1         |
|Documentary|CT0057    |Emerald Valley  |40819     

In [ ]:
subscriber_recent_window = (
    Window.partitionBy("subscriber_id")
    .orderBy(
        F.col("session_start_ts").desc_nulls_last(),
        F.col("session_id").desc()))

most_recent_session_per_subscriber_df = (
    sessions_timestamp_df
    .filter(F.col("subscriber_id").isNotNull())
    .withColumn(
        "row_num",
        F.row_number().over(subscriber_recent_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num"))

most_recent_session_per_subscriber_df.select(
    "subscriber_id", "session_id", "session_start_ts",
    "watch_minutes", "session_status").show(20, truncate=False)

+-------------+----------+-------------------+-------------+--------------+
|subscriber_id|session_id|session_start_ts   |watch_minutes|session_status|
+-------------+----------+-------------------+-------------+--------------+
|SUB00001     |WS00020822|2026-09-11 19:19:00|34           |Interrupted   |
|SUB00002     |WS00032744|2026-09-02 05:41:00|45           |Interrupted   |
|SUB00003     |WS00006110|2026-09-30 03:13:00|11           |Abandoned     |
|SUB00004     |WS00037660|2026-09-30 17:48:00|62           |Interrupted   |
|SUB00005     |WS00070939|2026-09-18 02:50:00|22           |Abandoned     |
|SUB00006     |WS00057918|2026-09-23 06:26:00|104          |Interrupted   |
|SUB00007     |WS00024658|2026-09-23 20:45:00|55           |Interrupted   |
|SUB00008     |WS00032330|2026-09-26 22:41:00|41           |Interrupted   |
|SUB00009     |WS00008916|2026-09-04 22:21:00|30           |Interrupted   |
|SUB00010     |WS00049685|2026-09-29 06:36:00|6            |Abandoned     |
|SUB00011   

In [ ]:
subscriber_session_window = (
    Window.partitionBy("subscriber_id")
    .orderBy("session_start_ts", "session_id"))

subscriber_watch_lag_df = (
    sessions_timestamp_df
    .filter(F.col("subscriber_id").isNotNull())
    .withColumn(
        "previous_watch_minutes",
        F.lag("watch_minutes", 1).over(subscriber_session_window))
    .withColumn(
        "watch_minutes_change",
        F.col("watch_minutes") - F.col("previous_watch_minutes")))

subscriber_watch_lag_df.select(
    "subscriber_id", "session_id", "session_start_ts",
    "watch_minutes", "previous_watch_minutes", "watch_minutes_change"
).orderBy("subscriber_id", "session_start_ts", "session_id").show(25, truncate=False)

+-------------+----------+-------------------+-------------+----------------------+--------------------+
|subscriber_id|session_id|session_start_ts   |watch_minutes|previous_watch_minutes|watch_minutes_change|
+-------------+----------+-------------------+-------------+----------------------+--------------------+
|SUB00001     |WS00064390|2026-01-02 17:32:00|58           |NULL                  |NULL                |
|SUB00001     |WS00045183|2026-01-05 04:51:00|78           |58                    |20                  |
|SUB00001     |WS00028484|2026-01-09 08:46:00|4            |78                    |-74                 |
|SUB00001     |WS00043555|2026-01-24 19:41:00|108          |4                     |104                 |
|SUB00001     |WS00041862|2026-02-18 07:43:00|79           |108                   |-29                 |
|SUB00001     |WS00059616|2026-02-25 07:38:00|18           |79                    |-61                 |
|SUB00001     |WS00042345|2026-03-13 07:34:00|44       

In [ ]:
main_sessions_count = sessions_df.count()
extra_sessions_count = extra_df.count()

combined_sessions_raw_df = sessions_df.unionByName(extra_df)
combined_sessions_count = combined_sessions_raw_df.count()

duplicate_session_ids_df = (
    combined_sessions_raw_df
    .groupBy("session_id")
    .count()
    .filter(F.col("count") > 1))

deduplicated_sessions_df = combined_sessions_raw_df.dropDuplicates(["session_id"])
unique_sessions_count = deduplicated_sessions_df.count()

print("Main batch rows:", main_sessions_count)
print("Additional batch rows:", extra_sessions_count)
print("Rows after union:", combined_sessions_count)
print("Repeated session_id values:", duplicate_session_ids_df.count())
print("Unique session rows after deduplication:", unique_sessions_count)


Main batch rows: 80000
Additional batch rows: 10000
Rows after union: 90000
Repeated session_id values: 600
Unique session rows after deduplication: 89400


In [ ]:
lazy_streaming_pipeline_df = (
    sessions_df
    .filter(
        (F.col("session_status") == "Completed") &
        F.col("subscriber_id").isNotNull())
    .join(
        subscribers_df.select("subscriber_id", "region"),
        on="subscriber_id",
        how="inner")
    .withColumn("watch_hours", F.col("watch_minutes") / 60.0)
    .groupBy("region")
    .agg(
        F.count("*").alias("completed_session_count"),
        F.round(F.sum("watch_hours"), 2).alias("completed_watch_hours"))
    .orderBy(F.col("completed_watch_hours").desc()))

lazy_streaming_pipeline_df.explain("formatted")

lazy_streaming_pipeline_df.show(truncate=False)

== Physical Plan ==
AdaptiveSparkPlan (14)
+- Sort (13)
   +- Exchange (12)
      +- HashAggregate (11)
         +- Exchange (10)
            +- HashAggregate (9)
               +- Project (8)
                  +- BroadcastHashJoin Inner BuildRight (7)
                     :- Project (3)
                     :  +- Filter (2)
                     :     +- Scan csv  (1)
                     +- BroadcastExchange (6)
                        +- Filter (5)
                           +- Scan csv  (4)


(1) Scan csv 
Output [3]: [subscriber_id#18, watch_minutes#22, session_status#26]
Batched: false
Location: InMemoryFileIndex [file:/content/watch_sessions_80000.csv]
PushedFilters: [IsNotNull(session_status), EqualTo(session_status,Completed), IsNotNull(subscriber_id)]
ReadSchema: struct<subscriber_id:string,watch_minutes:int,session_status:string>

(2) Filter
Input [3]: [subscriber_id#18, watch_minutes#22, session_status#26]
Condition : ((isnotnull(session_status#26) AND (session_status#26 = C

In [ ]:
completed_sessions_cached_df = sessions_df.filter(
    F.col("session_status") == "Completed").cache()

start = time.time()
completed_count_first = completed_sessions_cached_df.count()
count_first_seconds = time.time() - start

start = time.time()
completed_count_second = completed_sessions_cached_df.count()
count_second_seconds = time.time() - start

print("Completed sessions count:", completed_count_first)
print("Count action 1 seconds:", round(count_first_seconds, 4))
print("Count action 2 seconds:", round(count_second_seconds, 4))
print("Is cached:", completed_sessions_cached_df.is_cached)

def timed_completed_aggregation(df, label):
    start = time.time()
    result = (
        df.groupBy("session_status")
          .agg(
              F.count("*").alias("session_count"),
              F.round(F.avg("watch_minutes"), 2).alias("avg_watch_minutes")).count())
    elapsed = time.time() - start
    print(label, "| groups:", result, "| seconds:", round(elapsed, 4))
    return elapsed

aggregation_first_seconds = timed_completed_aggregation(
    completed_sessions_cached_df, "Cached aggregation 1")
aggregation_second_seconds = timed_completed_aggregation(
    completed_sessions_cached_df, "Cached aggregation 2")

completed_sessions_cached_df.unpersist()
print("Cache released.")

print("Observation: cache() marks the DataFrame for caching; the first action materializes it.")
print("Compare timings in this runtime. Cache benefits depend on workload and available memory.")

Completed sessions count: 6392
Count action 1 seconds: 1.3807
Count action 2 seconds: 0.1235
Is cached: True
Cached aggregation 1 | groups: 1 | seconds: 0.2174
Cached aggregation 2 | groups: 1 | seconds: 0.2197
Cache released.
Observation: cache() marks the DataFrame for caching; the first action materializes it.
Compare timings in this runtime. Cache benefits depend on workload and available memory.


In [ ]:
sessions_4k_persisted_df = (
    sessions_df
    .filter(F.col("stream_quality") == "4K")
    .persist(StorageLevel.MEMORY_AND_DISK))

four_k_count = sessions_4k_persisted_df.count()
four_k_avg_watch = sessions_4k_persisted_df.agg(
    F.round(F.avg("watch_minutes"), 2).alias("avg_4k_watch_minutes")).first()["avg_4k_watch_minutes"]

print("4K session count:", four_k_count)
print("Average watch minutes for 4K sessions:", four_k_avg_watch)

sessions_4k_persisted_df.unpersist()
print("4K persisted DataFrame released.")

4K session count: 7312
Average watch minutes for 4K sessions: 45.75
4K persisted DataFrame released.


In [ ]:
sessions_8_partitions_df = sessions_df.repartition(8)

session_partition_counts_df = (
    sessions_8_partitions_df
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .agg(F.count("*").alias("row_count"))
    .orderBy("partition_id"))

print("Partition count:", sessions_8_partitions_df.rdd.getNumPartitions())
session_partition_counts_df.show(20, truncate=False)

Partition count: 8
+------------+---------+
|partition_id|row_count|
+------------+---------+
|0           |10000    |
|1           |9999     |
|2           |10001    |
|3           |10000    |
|4           |10000    |
|5           |10000    |
|6           |10000    |
|7           |10000    |
+------------+---------+



In [ ]:
sessions_8_for_compare_df = sessions_df.repartition(8)
sessions_coalesce_2_df = sessions_8_for_compare_df.coalesce(2)
sessions_repartition_2_df = sessions_8_for_compare_df.repartition(2)

print("Starting partitions:", sessions_8_for_compare_df.rdd.getNumPartitions())
print("coalesce(2) partitions:", sessions_coalesce_2_df.rdd.getNumPartitions())
print("repartition(2) partitions:", sessions_repartition_2_df.rdd.getNumPartitions())

print("\nCOALESCE(2) PLAN")
sessions_coalesce_2_df.explain("formatted")

print("\nREPARTITION(2) PLAN")
sessions_repartition_2_df.explain("formatted")
task25_output_path = "/content/streaming_sessions_parquet"

(   sessions_repartition_2_df
    .write
    .mode("overwrite")
    .parquet(task25_output_path))

task25_part_files = [
    f for f in os.listdir(task25_output_path)
    if f.startswith("part-") and f.endswith(".parquet")]

print("Task 25 Parquet directory:", task25_output_path)
print("Part files:", task25_part_files)
print("Number of Parquet part files:", len(task25_part_files))

Starting partitions: 8
coalesce(2) partitions: 2
repartition(2) partitions: 2

COALESCE(2) PLAN
== Physical Plan ==
AdaptiveSparkPlan (7)
+- == Final Plan ==
   ResultQueryStage (5)
   +- Coalesce (4)
      +- ShuffleQueryStage (3), Statistics(sizeInBytes=12.8 MiB, rowCount=8.00E+4)
         +- Exchange (2)
            +- Scan csv  (1)
+- == Initial Plan ==
   Coalesce (6)
   +- Exchange (2)
      +- Scan csv  (1)


(1) Scan csv 
Output [12]: [session_id#17, subscriber_id#18, content_id#19, device_id#20, session_start#21, watch_minutes#22, completion_pct#23, buffering_seconds#24, stream_quality#25, session_status#26, data_used_mb#27, user_rating#28]
Batched: false
Location: InMemoryFileIndex [file:/content/watch_sessions_80000.csv]
ReadSchema: struct<session_id:string,subscriber_id:string,content_id:string,device_id:string,session_start:timestamp,watch_minutes:int,completion_pct:double,buffering_seconds:double,stream_quality:string,session_status:string,data_used_mb:double,user_rating:

**Final capstone - Streaming performance report**

In [ ]:
capstone_union_df = sessions_df.unionByName(extra_df)
capstone_sessions_df = capstone_union_df.dropDuplicates(["session_id"])

print("Capstone rows after union:", capstone_union_df.count())
print("Capstone rows after session_id deduplication:", capstone_sessions_df.count())

Capstone rows after union: 90000
Capstone rows after session_id deduplication: 89400


In [ ]:
capstone_sessions_timestamp_df = capstone_sessions_df.withColumn(
    "session_start_ts", F.to_timestamp("session_start"))

capstone_enriched_df = (
    capstone_sessions_timestamp_df
    .join(
        subscribers_df.select(
            "subscriber_id", "region", "plan_id", "subscriber_status"),
        on="subscriber_id",
        how="left").join(
        content_df.select(
            "content_id", "content_title", "genre"),
        on="content_id",
        how="left").join(
        devices_df.select("device_id", "device_type", "platform"),
        on="device_id",
        how="left").join(
        plans_df.select("plan_id", "plan_name", "monthly_fee_inr"),
        on="plan_id",
        how="left"))


In [ ]:
capstone_enriched_df = (
    capstone_enriched_df
    .withColumn("month", F.date_format("session_start_ts", "yyyy-MM"))
    .withColumn("watch_hours", F.col("watch_minutes") / 60.0)
    .withColumn(
        "completed_flag",
        F.when(F.col("session_status") == "Completed", 1).otherwise(0)))

capstone_enriched_df.select(
    "session_id", "month", "region", "genre", "content_title",
    "watch_minutes", "watch_hours", "completion_pct",
    "buffering_seconds", "data_used_mb", "session_status"
).show(10, truncate=False)

+----------+-------+------+-----------+---------------+-------------+-------------------+--------------+-----------------+------------+--------------+
|session_id|month  |region|genre      |content_title  |watch_minutes|watch_hours        |completion_pct|buffering_seconds|data_used_mb|session_status|
+----------+-------+------+-----------+---------------+-------------+-------------------+--------------+-----------------+------------+--------------+
|WS00000002|2026-07|West  |Animation  |Golden Archive |31           |0.5166666666666667 |56.4          |31.0             |422.7       |Interrupted   |
|WS00000003|2026-02|South |Drama      |Lost Echo      |29           |0.48333333333333334|63.0          |26.0             |370.0       |Interrupted   |
|WS00000004|2026-02|West  |Adventure  |Emerald Echo   |69           |1.15               |75.0          |2.0              |924.7       |Interrupted   |
|WS00000005|2026-04|South |Sports     |Midnight Echo  |36           |0.6                |23.4 

In [ ]:
monthly_streaming_report_df = (
    capstone_enriched_df
    .groupBy("month", "region", "genre")
    .agg(
        F.count("*").alias("session_count"),
        F.round(F.sum("watch_hours"), 3).alias("total_watch_hours"),
        F.round(
            F.sum("completed_flag") * 100.0 / F.count("*"),2).alias("completion_rate_pct"),
        F.round(F.avg("buffering_seconds"), 2).alias("avg_buffering_seconds"),
        F.round(F.sum("data_used_mb"), 2).alias("total_data_used_mb"))
    .orderBy("month", "region", "genre"))

monthly_streaming_report_df.show(100, truncate=False)

+-------+-------+-----------+-------------+-----------------+-------------------+---------------------+------------------+
|month  |region |genre      |session_count|total_watch_hours|completion_rate_pct|avg_buffering_seconds|total_data_used_mb|
+-------+-------+-----------+-------------+-----------------+-------------------+---------------------+------------------+
|2026-01|NULL   |Adventure  |37           |38.35            |8.11               |37.42                |44063.6           |
|2026-01|NULL   |Animation  |21           |13.017           |4.76               |29.68                |13333.2           |
|2026-01|NULL   |Comedy     |9            |4.417            |0.0                |30.78                |5805.3            |
|2026-01|NULL   |Documentary|22           |11.667           |9.09               |40.86                |13426.2           |
|2026-01|NULL   |Drama      |23           |17.217           |8.7                |34.09                |15182.3           |
|2026-01|NULL   

In [ ]:
genre_content_watch_hours_df = (
    capstone_enriched_df
    .groupBy("genre", "content_id", "content_title")
    .agg(
        F.round(F.sum("watch_hours"), 3).alias("total_watch_hours")))

capstone_title_rank_window = Window.partitionBy("genre").orderBy(
    F.col("total_watch_hours").desc())

capstone_top_titles_df = (
    genre_content_watch_hours_df
    .withColumn(
        "title_rank",
        F.dense_rank().over(capstone_title_rank_window))
    .filter(F.col("title_rank") <= 3)
    .orderBy("genre", "title_rank", "content_title"))

capstone_top_titles_df.show(100, truncate=False)

+-----------+----------+----------------+-----------------+----------+
|genre      |content_id|content_title   |total_watch_hours|title_rank|
+-----------+----------+----------------+-----------------+----------+
|Adventure  |CT0105    |Infinite Horizon|1210.8           |1         |
|Adventure  |CT0042    |Solar Memory    |1129.467         |2         |
|Adventure  |CT0048    |Lost Journey    |1127.85          |3         |
|Animation  |CT0106    |Last River      |969.3            |1         |
|Animation  |CT0114    |Emerald Circuit |963.083          |2         |
|Animation  |CT0100    |Golden Puzzle   |932.217          |3         |
|Comedy     |CT0019    |Bright Valley   |1173.8           |1         |
|Comedy     |CT0027    |Blue Horizon    |1151.25          |2         |
|Comedy     |CT0056    |Wild Canvas     |1136.233         |3         |
|Documentary|CT0077    |Fading Signal   |878.683          |1         |
|Documentary|CT0057    |Emerald Valley  |743.883          |2         |
|Docum

In [ ]:
final_streaming_report_cached_df = monthly_streaming_report_df.persist(
    StorageLevel.MEMORY_AND_DISK)

final_streaming_report_count = final_streaming_report_cached_df.count()
print("Final monthly report row count:", final_streaming_report_count)

Final monthly report row count: 432


In [ ]:
final_streaming_totals_df = final_streaming_report_cached_df.agg(
    F.sum("session_count").alias("total_sessions"),
    F.round(F.sum("total_watch_hours"), 3).alias("total_watch_hours"),
    F.round(F.avg("completion_rate_pct"), 2).alias("average_group_completion_rate_pct"),
    F.round(F.avg("avg_buffering_seconds"), 2).alias("average_group_buffering_seconds"),
    F.round(F.sum("total_data_used_mb"), 2).alias("total_data_used_mb"))

final_streaming_totals_df.show(truncate=False)

final_streaming_report_cached_df.unpersist()
print("Final streaming report cache released.")

capstone_streaming_output_path = "/content/streaming_monthly_performance_report_parquet"

(   monthly_streaming_report_df
    .repartition(3)
    .write
    .mode("overwrite")
    .parquet(capstone_streaming_output_path))

capstone_streaming_part_files = [
    f for f in os.listdir(capstone_streaming_output_path)
    if f.startswith("part-") and f.endswith(".parquet")]

print("Final capstone Parquet directory:", capstone_streaming_output_path)
print("Parquet part files:", capstone_streaming_part_files)
print("Number of Parquet part files:", len(capstone_streaming_part_files))

+--------------+-----------------+---------------------------------+-------------------------------+------------------+
|total_sessions|total_watch_hours|average_group_completion_rate_pct|average_group_buffering_seconds|total_data_used_mb|
+--------------+-----------------+---------------------------------+-------------------------------+------------------+
|89400         |67531.251        |7.89                             |36.1                           |7.49765326E7      |
+--------------+-----------------+---------------------------------+-------------------------------+------------------+

Final streaming report cache released.
Final capstone Parquet directory: /content/streaming_monthly_performance_report_parquet
Parquet part files: ['part-00001-392ad8dd-06e1-4215-9b52-85102d620553-c000.snappy.parquet', 'part-00002-392ad8dd-06e1-4215-9b52-85102d620553-c000.snappy.parquet', 'part-00000-392ad8dd-06e1-4215-9b52-85102d620553-c000.snappy.parquet']
Number of Parquet part files: 3
